In [0]:
# Project         : Procurement Analytics using Databricks & Power BI
# Layer           : Silver
# Notebook        : Silver_invoices
# Source          : invoices.csv
# Target          : procurement.silver.silver_invoices
#
# Author          : V R Mutyala
# Created Date    : 21-Jul-2026
# Last Modified   : 21-Jul-2026
#
# Description
# -----------
# This notebook loads Cleaned invoices master data into the Silver layer.
# It validates the source data, separates duplicate records, adds audit columns, and stores the results as Delta tables.

# ==============================================================================
# Business Objective
# ==============================================================================
#
# Read employee data from the Bronze layer, apply data cleansing,standardization, and business rules to create a trusted Silver Delta table.
# ==============================================================================

In [0]:
%run ../01_Config/Config

In [0]:
%run ../05_Helper_Functions/Helper_functions

In [0]:
# Import Libraries and Widgets
from pyspark.sql import DataFrame
from pyspark.sql.functions import (col, lit,current_timestamp,when,count,trim,coalesce,initcap,lower)
from pyspark.sql.types import (StructType, StructField, StringType,IntegerType,DoubleType,DecimalType)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import try_to_timestamp

In [0]:
# ============================================================
# Read Bronze Invoice Table
# ============================================================

bronze_invoices_df = read_delta(BRONZE_INVOICES)

preview(bronze_invoices_df,"Bronze invoices")

In [0]:
#============================================
# Create Sliver 
#===========================================
silver_invoices_df = bronze_invoices_df

In [0]:
# ============================================================
# Apply business transformations
# ============================================================

# Standardize invoice_status as Matched
silver_invoices_df = silver_invoices_df.withColumn(
    "invoice_status", 
    when(trim(col("invoice_status")) == "Match", "Matched")
    .otherwise(col("invoice_status")))

# Convert due_date to DateType
silver_invoices_df = silver_invoices_df.withColumn(
    "due_date",
    to_date(col("due_date"), "dd-MM-yyyy")
)

# Convert invoice_date  to DataType
silver_invoices_df = silver_invoices_df.withColumn(
    "invoice_date",
    to_date(col("invoice_date"), "dd-MM-yyyy")
)


In [0]:
silver_invoices_df.filter(col("invoice_id").isNull()).show(truncate=False)

silver_invoices_df.filter(trim(col("invoice_id")) == "").show(truncate=False)

In [0]:
# ============================================================
# Identify invalid invoice records
# NULL & Blank invoice_id
# NULL: invoice_date, po_id, po_amount_expected,invoice_amount, currency, due_date, invoice_status
# Zero: po_amount_expected, invoice_amount
# ============================================================
invalid_invoices = silver_invoices_df.filter(
    col("invoice_id").isNull()
    |(trim(col("invoice_id")) == "")
    |col("invoice_date").isNull()
    |col("po_id").isNull()
    |(col("po_amount_expected").isNull())|(col("po_amount_expected") == 0)
    |col("invoice_amount").isNull()
    |(col("invoice_amount") == 0)
    |col("currency").isNull()
    |col("due_date").isNull()
    |col("invoice_status").isNull()
    )

print(f"Invalid Invoice Records:{invalid_invoices.count()}")
display(invalid_invoices)

In [0]:
# ============================================================
# Add Audit Metadata
# ============================================================

invalid_invoices = (invalid_invoices.withColumn("audit_timestamp",current_timestamp())
    .withColumn("source_table",lit("Invoices"))
    .withColumn("pipeline_layer",lit("Silver"))
    .withColumn("issue_type",lit("Invalid Record")))

display(invalid_invoices)

In [0]:
# ============================================================
# Write Invalid Invoices to Audit Table
# ============================================================

if invalid_invoices.count() > 0:
    write_delta(invalid_invoices,AUDIT_INVALID_INVOICES,mode="overwrite")
    print("Invalid invoice records written.")
else:
    print("No invalid invoice records found.")

In [0]:
# ============================================================
# Remove Invalid Records
# ============================================================

silver_invoices_df = silver_invoices_df.filter(
     
    col("invoice_id").isNotNull() & (trim(col("invoice_id")) != "")

    & col("invoice_date").isNotNull()

    & col("po_id").isNotNull()

    & (col("po_amount_expected").isNotNull()) & (col("po_amount_expected") != 0)

    & (col("invoice_amount").isNotNull()) & (col("invoice_amount") != 0)

    & col("currency").isNotNull()

    & col("due_date").isNotNull()

    & col("invoice_status").isNotNull()
   )

display(silver_invoices_df)

In [0]:
# ============================================================
# Remove Duplicate Invoices
# ============================================================

window_spec = Window.partitionBy("invoice_id").orderBy("invoice_date")

silver_invoices_df = (silver_invoices_df.withColumn("row_num",row_number().over(window_spec))
                      .filter(col("row_num") == 1).drop("row_num"))
preview(silver_invoices_df,"Silver Invoices")


In [0]:
# ============================================================
# Apply Business Transformations
# ============================================================
from pyspark.sql.functions import coalesce, to_date, trim, col
from pyspark.sql.functions import expr

# Standardize invoice_status as Matched
silver_invoices_df = silver_invoices_df.withColumn(
    "invoice_status", 
    when(trim(col("invoice_status")) == "Match", "Matched")
    .otherwise(col("invoice_status")))

# Convert due_date to DateType
silver_invoices_df = silver_invoices_df.withColumn(
    "due_date",
    to_date(col("due_date"), "dd-MM-yyyy")
)

# Convert invoice_date  to DataType
silver_invoices_df = silver_invoices_df.withColumn(
    "invoice_date",
    to_date(col("invoice_date"), "dd-MM-yyyy")
)

display(silver_invoices_df)

In [0]:
print("Bronze Records :", bronze_invoices_df.count())
print("Invalid Records :", invalid_invoices.count())
print("Silver Before Duplicate Removal :", silver_invoices_df.count())

In [0]:
print("Silver After Duplicate Removal :", silver_invoices_df.count())

In [0]:
# ============================================================
# Write Silver Delta Table
# ============================================================

silver_invoices_df = (silver_invoices_df
                         .withColumn("silver_load_timestamp", current_timestamp())
                         .withColumn("pipeline_layer", lit("Silver"))
)

In [0]:
# ============================================================
# Write Silver Delta Table
# ============================================================

write_delta(df=silver_invoices_df,table_name=SILVER_INVOICES)

In [0]:
# ============================================================
# Validate Summary
# ============================================================

print("=" * 60)
print("Silver Invoices Load Completed Successfully")
print("=" * 60)

print(f"Bronze Records : {bronze_invoices_df.count()}")
print(f"Silver Records : {silver_invoices_df.count()}")
print(f"Duplicates Removed : {bronze_invoices_df.count() - silver_invoices_df.count()}")